# 8. Blind label audit

| | |
|---|---|
| **Stage** | 8 of 9 |
| **Purpose** | Estimate label noise in the Task 3 training table by re-labelling 60 sentence-aspect pairs blind, and compare the result with the original labels. |
| **Inputs** | `outputs/final_v4/training_table.csv`<br>`outputs/final_v4/t3_oof.npy`<br>`outputs/final_v4/data_hash.txt`<br>`scripts/aspect_taxonomy.py` |
| **Outputs** | `outputs/label_audit/audit_sheet_BLIND.xlsx`<br>`outputs/label_audit/audit_key_DO_NOT_OPEN_BEFORE_LABELLING.csv`<br>`outputs/label_audit/audit_labels_progress.csv`<br>`outputs/label_audit/label_audit_summary.csv (summary numbers, used by the dashboard)`<br>`outputs/label_audit/label_audit_scored_rows.csv (contains sentences; not uploaded)` |
| **Runtime** | CPU only. Step 2 is manual (about 40 to 60 minutes for 60 pairs). |
| **Depends on** | Stage 7 |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

## Why a blind audit

The error analysis in notebook 07 showed some labels that look doubtful (for example a clearly positive sentence labelled negative). To estimate how noisy the labels are, 60 sentence-aspect pairs are re-labelled **without seeing the original label, the model's prediction or the source**:
- 30 pairs drawn at random, which estimate the overall noise rate;
- 30 pairs where the cross-validated model disagreed most confidently with the label, which show what the extreme disagreements look like.

The pairs are shuffled so the type cannot be guessed from the position.

> **Limitation.** If the person re-labelling is also the original annotator, the audit measures consistency over time, not independent agreement, and probably understates label noise.

## Step 1 · Build the blind audit sheet

Reads the saved training table and out-of-fold predictions, selects the 60 pairs and writes the blind sheet and a separate key. **Do not open the key file before labelling.** Writes only to `outputs/label_audit/`.

In [ ]:
# ===== Blind label audit, step 1: build the audit sheet (CPU only; reads saved files, writes ONLY to outputs/label_audit/) =====
import os, sys, json, hashlib
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
sys.path.insert(0, f"{PROJECT_ROOT}/scripts")
from aspect_taxonomy import ASPECT_TAXONOMY
FINAL = f"{PROJECT_ROOT}/outputs/final_v4"
AUD = f"{PROJECT_ROOT}/outputs/label_audit"
os.makedirs(AUD, exist_ok=True)
SHEET = f"{AUD}/audit_sheet_BLIND.xlsx"
KEY = f"{AUD}/audit_key_DO_NOT_OPEN_BEFORE_LABELLING.csv"
N_RANDOM, SEED = 30, 7                                  # 30 most confident disagreements + 30 random pairs
tt = pd.read_csv(f"{FINAL}/training_table.csv")
oof = np.load(f"{FINAL}/t3_oof.npy")
assert len(tt) == len(oof), f"training table ({len(tt)}) and out-of-fold predictions ({len(oof)}) have different lengths"
h = hashlib.sha256("|".join(tt["sentence_uid"].astype(str) + tt["aspect"] + tt["label"] + tt["source"]).encode()).hexdigest()[:16]
hp = f"{FINAL}/data_hash.txt"
if os.path.exists(hp) and open(hp).read().strip() != h:
    print("WARNING: the training table does not match the one the predictions were made for; the 'disagreement' rows may be wrong.")
else:
    print("training table and predictions are consistent")

SENT = ["negative", "neutral", "positive"]
tt["y"] = tt["label"].map({l: i for i, l in enumerate(SENT)})
tt["pred"] = oof.argmax(1); tt["p_neg"] = oof[:, 0]
tt["model_label"] = tt["pred"].map(dict(enumerate(SENT)))

# the same 30 rows that the error analysis showed: most confident missed complaints and false alarms
missed = tt[(tt["y"] == 0) & (tt["pred"] != 0)].sort_values("p_neg").head(15)
alarm = tt[(tt["y"] != 0) & (tt["pred"] == 0)].sort_values("p_neg", ascending=False).head(15)
susp = pd.concat([missed, alarm]).assign(sample_type="most_confident_disagreement")
rest = tt.drop(index=susp.index)
rnd = rest.sample(min(N_RANDOM, len(rest)), random_state=SEED).assign(sample_type="random")
aud = pd.concat([susp, rnd]).sample(frac=1, random_state=11).reset_index(drop=True)     # shuffled, so type cannot be guessed from position
aud["item_id"] = np.arange(1, len(aud) + 1)

meta = lambda a: ASPECT_TAXONOMY.get(a, {})
sheet = pd.DataFrame({
    "item_id": aud["item_id"], "game": aud["game"],
    "aspect": [str(meta(a).get("label", a)) for a in aud["aspect"]],
    "aspect_meaning": [str(meta(a).get("description", "")) for a in aud["aspect"]],
    "sentence_text": aud["sentence_text"],
    "your_label": "", "your_note": ""})
key = aud[["item_id", "sample_type", "sentence_uid", "aspect", "label", "source", "model_label", "p_neg"]].rename(columns={"label": "original_label"})
key["p_neg"] = key["p_neg"].round(3)
key.to_csv(KEY, index=False)

with pd.ExcelWriter(SHEET, engine="openpyxl") as xw:
    sheet.to_excel(xw, index=False, sheet_name="audit")
    ws = xw.sheets["audit"]
    from openpyxl.worksheet.datavalidation import DataValidation
    from openpyxl.styles import Alignment, Font
    dv = DataValidation(type="list", formula1='"negative,neutral,positive,not_about_aspect,unsure"', allow_blank=True, showErrorMessage=True,
                        errorTitle="Pick one", error="Use: negative, neutral, positive, not_about_aspect or unsure")
    ws.add_data_validation(dv); dv.add(f"F2:F{len(sheet) + 1}")
    for col, w in zip("ABCDEFG", (8, 12, 24, 46, 80, 18, 40)):
        ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    for c in ws[1]:
        c.font = Font(bold=True)
    ws.freeze_panes = "A2"

print(f"\nAudit sheet: {len(sheet)} rows -> {SHEET}")
print(f"Key (do not open yet): {KEY}")
print("Sample sizes:", key["sample_type"].value_counts().to_dict())
print("\nFirst 5 rows of the sheet (this is what you will see):")
print(sheet.head(5)[["item_id", "game", "aspect", "sentence_text"]].to_string(index=False))


## Step 2 · Label the pairs inside the notebook

One pair is shown at a time; click a button to save the label and move on. Progress is saved after every click, and rerunning the cell resumes at the first unlabelled pair.

**Labelling rules**
- Judge only how the **sentence** feels about the **named aspect**; do not guess what the model would say.
- `negative`: complaint or dissatisfaction with that aspect. `positive`: praise. `neutral`: factual or no clear polarity.
- `not_about_aspect`: the sentence does not actually mention that aspect. `unsure`: you cannot decide.
- If a sentence is recognised from earlier analysis, label it from the sentence alone and write `recognised` in the note field.

If the buttons do not appear, run `from google.colab import output; output.enable_custom_widget_manager()` and rerun the cell.

In [ ]:
# ===== Blind label audit, step 2: label inside the notebook (progress is saved after every click; rerun this cell to resume) =====
import os, html, datetime
import pandas as pd
import ipywidgets as W
from IPython.display import display
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
AUD = f"{PROJECT_ROOT}/outputs/label_audit"
SHEET = f"{AUD}/audit_sheet_BLIND.xlsx"                  # the blind sheet: no original labels, no model predictions
PROGRESS = f"{AUD}/audit_labels_progress.csv"            # your labels, saved after every click
LABELS = [("negative", "danger"), ("neutral", ""), ("positive", "success"), ("not_about_aspect", "info"), ("unsure", "warning")]

items = pd.read_excel(SHEET)[["item_id", "game", "aspect", "aspect_meaning", "sentence_text"]].fillna("").reset_index(drop=True)
N = len(items)
store = {}
if os.path.exists(PROGRESS):
    p = pd.read_csv(PROGRESS, keep_default_na=False)
    store = {int(r.item_id): (str(r.your_label), str(r.your_note)) for r in p.itertuples()}
ids = [int(x) for x in items["item_id"]]
state = {"i": next((k for k in range(N) if ids[k] not in store), 0)}

def save():
    rows = [{"item_id": k, "your_label": v[0], "your_note": v[1], "saved_at": datetime.datetime.now().isoformat(timespec="seconds")} for k, v in sorted(store.items())]
    tmp = PROGRESS + ".tmp"
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, PROGRESS)

head, body, status = W.HTML(), W.HTML(), W.HTML()
note = W.Text(placeholder="optional note (e.g. 'recognised', or why you chose this)", layout=W.Layout(width="75%"))
buttons = {lab: W.Button(description=lab, button_style=sty, layout=W.Layout(width="170px", height="42px")) for lab, sty in LABELS}
b_back, b_skip, b_first = W.Button(description="< Back"), W.Button(description="Skip >"), W.Button(description="Go to first unlabelled")

def render():
    i = state["i"]; r = items.iloc[i]; iid = ids[i]
    done = len(store)
    head.value = f"<h3>Item {i + 1} of {N} &nbsp;&nbsp;<small>({done} labelled)</small></h3>"
    cur = f"<div style='color:#2E5597'>Your current label: <b>{html.escape(store[iid][0])}</b> (click another button to change it)</div>" if iid in store else ""
    body.value = (f"<div style='font-size:15px'><b>{html.escape(str(r['game']))}</b> &nbsp;|&nbsp; Aspect: <b>{html.escape(str(r['aspect']))}</b></div>"
                  f"<div style='color:#666;font-size:12px;margin:2px 0 10px 0'>{html.escape(str(r['aspect_meaning']))}</div>"
                  f"<div style='font-size:19px;line-height:1.5;padding:12px;border:1px solid #ccc;border-radius:6px'>{html.escape(str(r['sentence_text']))}</div>"
                  f"<div style='margin-top:8px'>How does this sentence feel about this aspect?</div>{cur}")
    note.value = store.get(iid, ("", ""))[1]
    if done == N:
        counts = pd.Series([v[0] for v in store.values()]).value_counts().to_dict()
        status.value = f"<div style='color:green;font-size:16px'><b>All {N} items are labelled.</b> {counts}. You can now run the scoring cell.</div>"
    else:
        status.value = f"<div style='color:#666'>{N - done} left. Progress is saved automatically.</div>"

def make_handler(lab):
    def h(_):
        store[ids[state["i"]]] = (lab, note.value.strip())
        save()
        if state["i"] < N - 1:
            state["i"] += 1
        render()
    return h
for lab, b in buttons.items():
    b.on_click(make_handler(lab))
def go(delta):
    def h(_):
        state["i"] = max(0, min(N - 1, state["i"] + delta)); render()
    return h
def go_first(_):
    state["i"] = next((k for k in range(N) if ids[k] not in store), N - 1); render()
b_back.on_click(go(-1)); b_skip.on_click(go(1)); b_first.on_click(go_first)

render()
display(W.VBox([head, body, W.HBox(list(buttons.values())), note, W.HBox([b_back, b_skip, b_first]), status]))


## Step 3 · Score the audit (run only after all 60 pairs are labelled)

Compares the blind labels with the original labels. The cell stops with a clear message if any pair is unlabelled.

**How to read the output**
- *share differing* with its 95% interval, for the random sample and for the confident-disagreement sample separately.
- *kappa* between the original and the blind labels (three classes).
- The cross-tabulation shows which classes are confused; flips between negative and positive are the serious ones.
- *differs AND agrees with the model* counts the doubtful labels where the blind label matches the model, that is, probable label errors rather than model errors.
- The estimate for the whole training table assumes the random sample is representative; with 30 pairs the interval is wide.

In [ ]:
# ===== Blind label audit, step 2: score your labels against the original ones (CPU only) =====
import os, sys
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
AUD = f"{PROJECT_ROOT}/outputs/label_audit"
PROGRESS = f"{AUD}/audit_labels_progress.csv"            # written by the labelling cell
FILLED = f"{AUD}/audit_sheet_FILLED.xlsx"                # optional alternative: a sheet filled in outside the notebook
KEY = f"{AUD}/audit_key_DO_NOT_OPEN_BEFORE_LABELLING.csv"
SENT = ["negative", "neutral", "positive"]
OUT_SUMMARY, OUT_ROWS = f"{AUD}/label_audit_summary.csv", f"{AUD}/label_audit_scored_rows.csv"

if os.path.exists(PROGRESS):
    fl = pd.read_csv(PROGRESS, keep_default_na=False); print("labels read from", PROGRESS)
else:
    fl = pd.read_excel(FILLED); print("labels read from", FILLED)
key = pd.read_csv(KEY)
def norm(x):
    t = str(x).strip().lower()
    if t in ("", "nan", "none"): return ""
    if t.startswith("neg"): return "negative"
    if t.startswith("neu"): return "neutral"
    if t.startswith("pos"): return "positive"
    if t.startswith("not"): return "not_about_aspect"
    if t.startswith("uns"): return "unsure"
    return "?" + t
m = key.merge(fl[["item_id", "your_label", "your_note"]], on="item_id", how="left")
m["your_label"] = m["your_label"].fillna("").map(norm)
bad = m[m["your_label"].str.startswith("?")]
blank = m[m["your_label"] == ""]
assert len(bad) == 0, f"unrecognised labels in items {bad['item_id'].tolist()}: {bad['your_label'].tolist()}"
assert len(blank) == 0, f"{len(blank)} items are still unlabelled (item_id {blank['item_id'].tolist()[:15]} ...). Finish labelling before scoring."
m["comparable"] = m["your_label"].isin(SENT)
m["same_as_original"] = m["your_label"] == m["original_label"]
m["same_as_model"] = m["your_label"] == m["model_label"]

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)

rows = []
for name, g in list(m.groupby("sample_type")) + [("all audited", m)]:
    n = len(g); k = int((~g["same_as_original"]).sum()); lo, hi = wilson(k, n)
    comp = g[g["comparable"]]
    kappa = cohen_kappa_score(comp["original_label"], comp["your_label"]) if comp["original_label"].nunique() > 1 and len(comp) > 5 else np.nan
    rows.append({"sample": name, "n": n, "your label differs from original": k, "share differing": round(k / n, 3), "95% CI": f"[{lo:.2f}, {hi:.2f}]",
                 "of which 'not about aspect' or 'unsure'": int(g["your_label"].isin(["not_about_aspect", "unsure"]).sum()),
                 "differs AND agrees with the model": int((~g["same_as_original"] & g["same_as_model"]).sum()),
                 "kappa (original vs yours, 3 classes)": round(float(kappa), 2) if pd.notna(kappa) else np.nan})
summ = pd.DataFrame(rows)
print("=== How often does your blind label differ from the original label? ===")
print(summ.to_string(index=False))

print("\n=== Original label (rows) vs your blind label (columns) ===")
print(pd.crosstab(m["original_label"], m["your_label"]).to_string())
for name, g in m.groupby("sample_type"):
    print(f"\n--- {name}: original vs yours ---")
    print(pd.crosstab(g["original_label"], g["your_label"]).to_string())

rnd = m[(m["sample_type"] == "random") & m["comparable"]]
if len(rnd):
    acc_o = float(((rnd["model_label"] == "negative") == (rnd["original_label"] == "negative")).mean())
    acc_y = float(((rnd["model_label"] == "negative") == (rnd["your_label"] == "negative")).mean())
    print(f"\nOn the {len(rnd)} random pairs, the model's negative-vs-rest accuracy is {acc_o:.2f} against the original labels and {acc_y:.2f} against your blind labels.")
r = summ[summ["sample"] == "random"].iloc[0]
rate = r["share differing"]; lo, hi = wilson(int(r["your label differs from original"]), int(r["n"]))
print(f"\nIf the random sample is representative, about {rate:.0%} of the labels (95% CI {lo:.0%} to {hi:.0%}) would change on a blind re-label, "
      f"i.e. roughly {rate * 757:.0f} of 757 pairs (range {lo * 757:.0f} to {hi * 757:.0f}). n=30 gives a wide interval.")
print("\nCaution: you are the original annotator, so this measures your own consistency over time, not independent agreement; "
      "it is likely to understate the true noise.")
m.drop(columns=["sentence_uid"]).to_csv(OUT_ROWS, index=False); summ.to_csv(OUT_SUMMARY, index=False)
print("\nSaved:", OUT_SUMMARY, "and", OUT_ROWS)
